In [1]:
import pandas as pd 

df = pd.read_csv('/kaggle/input/data-cleaning-part-2/final_annotated_dataset.csv')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45446 entries, 0 to 45445
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    45446 non-null  object
 1   labels  45446 non-null  object
dtypes: object(2)
memory usage: 710.2+ KB


In [3]:
label_names = df['labels'].str.split(',').explode().str.strip().unique().tolist()
len(label_names)

49

In [4]:
from collections import Counter

lbl_counter = Counter()

for row in df['labels']:
    for label in row.split(','):
        lbl_counter[label.strip()] +=1

item_df = pd.DataFrame(lbl_counter.items(), columns =['label', 'count']).sort_values(by='count', ascending=False)


In [5]:
threshold = int(len(df)*0.005)
print(threshold)
rare_labels = item_df[item_df['count'] < threshold]
print(rare_labels['label'].to_list())
print(rare_labels['count'].sum())

227
['বিনোদন', 'চটগ্রাম', 'বিশেষ প্রতিবেদন', 'ফ্যাশন']
654


In [6]:
def clean_rare_labels(labels_str):
    lbl = [x.strip() for x in labels_str.split(',')]
    cleared_lbl = [l for l in lbl if l not in rare_labels['label'].to_list()]
    return ', '.join(cleared_lbl)

df['labels'] = df['labels'].apply(clean_rare_labels)

In [7]:
label_names = df['labels'].str.split(',').explode().str.strip().unique().tolist()
len(label_names)

45

In [8]:
empty_rows = df[
    df['labels'].isna() | (df['labels'].str.strip() == "")
]

print("Empty rows:", len(empty_rows))


Empty rows: 0


In [9]:
one_label_rows = df[df['labels'].str.count(',') == 0]
len(one_label_rows)


265

In [10]:
one_label_rows['labels'].unique()

array(['আবহাওয়া', 'আইন', 'শিক্ষা', 'ক্রীড়া', 'ধর্ম', 'রাজনীতি',
       'আন্তর্জাতিক', 'দুর্ঘটনা', 'উৎসব', 'সরকারি-নীতি', 'প্রযুক্তি',
       'জেলা সংবাদ', 'ঢাকা', 'বাণিজ্য'], dtype=object)

In [11]:
df = df.drop(one_label_rows.index).reset_index(drop=True)


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45181 entries, 0 to 45180
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    45181 non-null  object
 1   labels  45181 non-null  object
dtypes: object(2)
memory usage: 706.1+ KB


In [13]:
df.to_csv('bnmc_dataset.csv')

In [14]:
#huggingface_dataset

from datasets import Dataset, DatasetDict

initial_dataset = Dataset.from_pandas(df, preserve_index = False)

ini_split = initial_dataset.train_test_split(test_size = 0.2)
final_split = ini_split['test'].train_test_split(test_size = 0.5)

final_dataset = DatasetDict(
    {
        'train': ini_split['train'],
        'test' : final_split['test'],
        'val' : final_split['train']
    }
)


In [15]:
final_dataset.save_to_disk('bnmc_hf_dataset')

Saving the dataset (0/1 shards):   0%|          | 0/36144 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4519 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4518 [00:00<?, ? examples/s]